# 2. El Ritual de Calibración (Métrica Real)

Para que un sistema de visión pase de ser cualitativo a cuantitativo, necesitamos conocer los parámetros intrínsecos de la HQ Camera.  
- El Reto: Usa un patrón de tablero de ajedrez (chessboard) para obtener la <font color=#09FF >Matriz de la Cámara </font>  y los <font color=#09FF >Coeficientes de Distorsión</font>. 

- Por qué importa: Sin este paso, las distancias que midas serán solo estimaciones. La calibración corrige la distorsión del lente, permitiendo medidas milimétricas necesarias para una tesis de física.  

Para avanzar de una detección cualitativa a una metrología de precisión en tu tesis de física, el Ejercicio 2 es fundamental. En el marco teórico ya mencione el Modelo de Cámara Estenopeica (Pinhole) , pero para que la ecuación $D = \frac{W \cdot f}{P}$ sea exacta , necesitamos corregir las aberraciones ópticas del lente de tu Raspberry Pi HQ Camera.

La calibración geométrica permite obtener la Matriz Intrínseca de la cámara y los Coeficientes de Distorsión. Esto asegura que las distancias medidas sean reales y no meras estimaciones afectadas por la curvatura del lente.

 # 1. Preparación: El Patrón de Calibración
 
 Necesitamos imprimir un tablero de ajedrez (Chessboard). Es vital que el patrón esté pegado sobre una superficie totalmente plana (como un vidrio o madera rígida) para evitar errores de curvatura adicionales.
 
 [!TIP]Cuenta el número de "esquinas internas". Por ejemplo, un tablero de $8 \times 6$ cuadros tiene $7 \times 5$ esquinas internas. Estos son los puntos que OpenCV buscará.
 

In [1]:
import cv2
import numpy as np

# =====================================================
# CONFIGURACIÓN DEL TABLERO
# =====================================================

#Número REAL de cuadros
num_squares_x = 8
num_squares_y = 6

#Tamaño de cada cuadro en pixeles
square_size = 100

#margenes blancos
margin = 100

# =====================================================
# CREAR IMAGEN
# =====================================================

width = num_squares_x * square_size + 2 * margin
height = num_squares_y * square_size + 2 * margin

board = np.ones((height,width),dtype=np.uint8)*255

# =====================================================
# DIBUJAR CUADROS NEGROS
# =====================================================

for y in range(num_squares_y):
    for x in range(num_squares_x):
        #Alternar colores
        if (x + y) % 2 == 0:
            x1 = margin + x * square_size
            y1 = margin + y * square_size

            x2 = x1 + square_size
            y2 = y1 + square_size

            cv2.rectangle(board,(x1,y1),(x2,y2),0,-1)

# =====================================================
# GUARDAR
# =====================================================

cv2.imwrite("checkerboard_8x6.png", board)

print("Tablero generado correctamente.")


Tablero generado correctamente.


 # 2. Paso A: Captura de Imágenes
 Debes tomar entre 15 y 25 fotografías del tablero desde diferentes ángulos y distancias. Asegúrate de que el tablero cubra todas las áreas del frame (esquinas, centro, bordes).

In [ ]:
import cv2
import numpy as np
import glob

#Definir dimensiones del tablero (esquinas internas)
CHECKBOARD = (7,5)
criteria = (cv2-cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,30,0.001)

# vectores para puntos 3D reales y 2D en imagen
objpoints = []
imgpoints = []

#Preparar puntos 3D (0,0,0), (1, 0,0), (2,0,0)...
objp = np.zeros((1,CHECKBOARD[0]*CHECKBOARD[1],3),np.float32)
objp[0,:,:2] = np.mgrid[0:CHECKBOARD[0],0:CHECKBOARD[1]].T.reshape(-1,2)

#Cargamos imagenes captuiradas
images = glob.glob('calib_*.jpg')

